# edgar-extract — LoRA fine-tune (ücretsiz T4)

Adım ②. Ayrıntılı gerekçeler: repodaki `COLAB.md`.

**Runtime → Change runtime type → T4 GPU** seçili olmalı. Hücreleri sırayla çalıştırın.

İki yerde durup çıktıya bakmanız isteniyor (5. ve 6. hücre). Oralar, üç saatlik bir
koşuyu boşa harcamamak için var.

In [ ]:
# 1) GPU DOĞRULAMA
!nvidia-smi --query-gpu=name,memory.total,compute_cap --format=csv

# Beklenen: Tesla T4, 15360 MiB, 7.5
# 7.5 = Turing = bf16 YOK. train_lora.py bunu kendisi görüp fp16 seçiyor;
# sizin bir şey yapmanız gerekmiyor. Rehberlerden kopyaladığınız BAŞKA bir
# script bf16=True (TRL varsayılanı) ile burada patlarsa sebebi budur.

In [ ]:
# 2) KURULUM — sürümler SABİT
# TRL'in SFT API'si bu proje yazılırken değişti (max_seq_length -> max_length,
# varsayılan 1024). Eski adı veren script HATA VERMEZ, sessizce her örneği keser.
!pip install -q transformers==5.14.1 trl==1.9.2 peft==0.20.0 datasets==5.0.1 accelerate==1.14.0 bitsandbytes

In [ ]:
# 3) PAKETİ YÜKLEYİN — yerelde `python src/pack_colab.py` ile üretilen
#    data/processed/colab-bundle.zip dosyasını seçin.
import os, zipfile
from google.colab import files

os.makedirs('/content/edgar-extract', exist_ok=True)
up = files.upload()
name = list(up)[0]
with zipfile.ZipFile(name) as z:
    z.extractall('/content/edgar-extract')
os.chdir('/content/edgar-extract')

# Paket TAM mı? Eksik dosyayı Colab'de fark etmek oturumu boşa harcar.
import json, pathlib
for f in ['src/prompt.py','src/train_lora.py','src/predict.py',
          'data/processed/sft_train.jsonl','data/processed/sft_dev.jsonl',
          'data/processed/sft_test.jsonl','data/processed/token_report.json']:
    print(('VAR ' if pathlib.Path(f).exists() else 'EKSİK'), f)
r = json.load(open('data/processed/token_report.json'))
print('\nölçülen taban seq_len:', r['min_seq_len_no_truncation'], '-> önerilen', r['recommended_seq_len'])

---
## 🔴 DURAK 1 — smoke test

Aşağıdaki hücre öğrenmek için değil. Baktığınız **tek satır**:

```
kayip maskesi: ... token'in ...'sinde kayip hesaplaniyor (%4.7)
```

**~%5 olmalı.** Bu, kaybın yalnız JSON hedefinde hesaplandığı anlamına gelir — 700
token'lık talimat maskeleniyor. Oran yarıdan büyükse `completion_only_loss`
çalışmıyordur ve model **talimatı üretmeyi** öğrenir. O durumda devam etmeyin.

In [ ]:
# 4) SMOKE — 135M model, 2 adım, ~2 dakika
!python src/train_lora.py --smoke

---
## 🔴 DURAK 2 — VRAM ölçümü

Gerçek modelle birkaç adım koşup **tepe VRAM**'i ölçer. OOM'u üç saatlik koşunun
10. dakikasında değil, burada görün.

Sığmazsa sırayla: `--rank 8` → `--4bit`.
**`--max-length`'i DÜŞÜRMEYİN** — 3072 ölçülmüş taban, altına inmek örnekleri keser
ve kesilen yer tam da çıkarılacak alanların bulunduğu bölgedir.

In [ ]:
# 5) VRAM SONDASI — kısa koşu
!python src/train_lora.py --epochs 0.05

In [ ]:
# 6) GERÇEK KOŞU — 99 eğitim örneği, 3 epoch
# Adaptör her epoch sonunda kaydediliyor; oturum koparsa baştan başlamazsınız.
#
# 🔴 loss 'nan' olursa İLK ŞÜPHELİ fp16'dır (T4'te bf16 yok), model ya da veri
#    değil. Veriyi kurcalamadan önce --lr 5e-5 deneyin.
!python src/train_lora.py

---
## Sıra önemli: önce DEV, sonra TEST

`dev` (25 kayıt) model seçimi için — epoch/lr/checkpoint burada kıyaslanır.
`test` (36 kayıt) **bir kez** bakılır; şimdi bakılırsa dondurulmuş baseline sayısı
(%27,8) anlamını kaybeder.

Bu yüzden önce yalnız dev tahminleri üretilir, indirilir, **yerelde** ölçülür
(altın etiketler pakette yok — bilerek). Yapılandırma kesinleştikten sonra test.

In [ ]:
# 7) DEV tahminleri (model seçimi)
!python src/predict.py --adapter models/lora-qwen2.5-1.5b --split dev

from google.colab import files
files.download('data/processed/preds_ft_dev.jsonl')

In [ ]:
# 8) TEST tahminleri — YAPILANDIRMA KESİNLEŞTİKTEN SONRA
!python src/predict.py --adapter models/lora-qwen2.5-1.5b --split test

# Üçüncü yarışmacı: prompted BÜYÜK model, aynı T4'te 4-bit.
# Sığmazsa Qwen2.5-3B-Instruct'a düşün ve raporda öyle yaz.
!python src/predict.py --model Qwen/Qwen2.5-7B-Instruct --4bit --split test -o data/processed/preds_prompted7b_test.jsonl

from google.colab import files
files.download('data/processed/preds_ft_test.jsonl')
files.download('data/processed/preds_prompted7b_test.jsonl')

In [ ]:
# 9) ADAPTÖRÜ İNDİRİN — artifact'ın kendisi, birkaç MB
!zip -qr adapter.zip models/lora-qwen2.5-1.5b
from google.colab import files
files.download('adapter.zip')

---
## Sonra: yerelde ölçüm

İndirilen dosyaları `data/processed/` altına koyup:

```bash
python src/evaluate.py data/processed/preds_ft_dev.jsonl --split dev      # seçim

python src/evaluate.py \
  data/processed/preds_regex_test.jsonl \
  data/processed/preds_ft_test.jsonl \
  data/processed/preds_prompted7b_test.jsonl \
  --json-out data/processed/eval_all.json
```

**Aşılması gereken çubuk** (kural-tabanlı, test): tam kayıt **%27,8** · zor vaka
**%81,6** · doğru abstention %90,2 · şema geçerliliği %100.

⚠️ Regex'in **dev** skoru (%68,0) kirlidir — kurallar dev oyulmadan önce tüm train
üzerinde ayarlandı. Fine-tuned modelin dev skoruyla karşılaştırmayın; karşılaştırma
yeri yalnız test.

Fine-tune bunları geçmezse sonuç **"fine-tune bu görevde kural-tabanlıyı yenmedi"**
olur ve öyle raporlanır. Ölçümün amacı kazanmak değil, öğrenmek.